İstatistiksel karşılaştırma hücresi dış değerlendirmelerden sonra çalıştırılır.


# 28B — FinancialBERT test ve değerlendirme

Bu notebook, projedeki FinBERT akışının tamamını `ahmedrachid/FinancialBERT-Sentiment-Analysis` hazır sentiment sınıflandırıcısı için tekrarlar:

1. Hazır model ile zero-shot baseline ve hata analizi
2. Aynı plain-sentiment train/validation/test splitleriyle hedef veri fine-tuning
3. Üçlü consensus train/validation/internal-test splitleriyle fine-tuning
4. Validation seçimi, epoch checkpointleri ve kaldığı yerden devam
5. İç test ve üç dış test setinde örnek-bazlı tahminler, raporlar ve confusion matrix
6. Hazır / hedefe fine-tune / consensus fine-tune karşılaştırması

Model kartına göre model Financial PhraseBank üzerinde üç sınıflı sentiment için fine-tune edilmiştir. Bu notebook hazır classifier head'i kullanır; yalnızca fine-tuning aşamalarında proje label sırasına hizalama yapar.

In [7]:
from pathlib import Path
import os, sys, json, gc, inspect, random
import numpy as np
import pandas as pd
import torch
from torch import nn
from datasets import Dataset
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
    f1_score, classification_report, confusion_matrix)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
from thesis_utils import PROJECT_ROOT, DATA_ROOT, paths

os.environ['WANDB_DISABLED'] = 'true'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
LABELS = list(LABEL2ID)
# Tez protokolündeki doğru başlangıç modeli; sentiment-head bu checkpoint'ten alınır.
MODEL_NAME = 'ahmedrachid/FinancialBERT'
RANDOM_STATE, MAX_LENGTH = 42, 128
EPOCHS, TRAIN_BATCH_SIZE, EVAL_BATCH_SIZE = 4, 16, 32
LEARNING_RATE, WEIGHT_DECAY, SAVE_TOTAL_LIMIT = 2e-5, 0.01, 3
USE_CLASS_WEIGHTS, FORCE_RETRAIN = True, False
# Her aşama kendi marker'ını yazar; notebook kapanırsa yalnız eksik aşama yeniden çalışır.
CHECKPOINT_SCHEMA_VERSION = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

RUN_ROOT = paths.MODEL_CHECKPOINT_ROOT / 'financialbert_full_pipeline_financialbert_seed42_v2'
TARGET_ROOT = RUN_ROOT / 'target_dataset'
CONSENSUS_ROOT = RUN_ROOT / 'consensus'
RESULT_ROOT = paths.MODEL_RESULTS_ROOT / 'financialbert_full_pipeline_financialbert_seed42_v2'
for p in [TARGET_ROOT, CONSENSUS_ROOT, RESULT_ROOT]: p.mkdir(parents=True, exist_ok=True)

def seed_all(seed=RANDOM_STATE):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_all()
print('Device:', DEVICE)
print('Model:', MODEL_NAME)
print('Run root:', RUN_ROOT)
RUN_MANIFEST = RUN_ROOT / 'run_manifest.json'
EXPECTED_MANIFEST = {'model_name': MODEL_NAME, 'seed': RANDOM_STATE, 'schema': CHECKPOINT_SCHEMA_VERSION}
if RUN_MANIFEST.exists():
    saved_manifest = json.loads(RUN_MANIFEST.read_text(encoding='utf-8'))
    if saved_manifest != EXPECTED_MANIFEST:
        raise RuntimeError(f'Run manifest mismatch; refusing to resume: {saved_manifest}')
else:
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    RUN_MANIFEST.write_text(json.dumps(EXPECTED_MANIFEST, indent=2), encoding='utf-8')


Device: cpu
Model: ahmedrachid/FinancialBERT
Run root: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\financialbert_full_pipeline_financialbert_seed42_v2


In [8]:
# Ortak yardımcılar: veri, label hizalama, metrik ve checkpoint destekli Trainer
def norm_label(x):
    s = str(x).lower().strip().replace(' ', '_')
    aliases = {'label_0':'positive','label_1':'negative','label_2':'neutral',
               'positive':'positive','negative':'negative','neutral':'neutral',
               'pos':'positive','neg':'negative','neu':'neutral'}
    return aliases.get(s, s)

def align_classifier(model):
    if not hasattr(model, 'classifier'):
        raise AttributeError('FinancialBERT classifier katmanı bulunamadı.')
    raw = {int(k): norm_label(v) for k, v in model.config.id2label.items()}
    layer = model.classifier
    # ahmedrachid/FinancialBERT iki çıkışlı head ile geliyor; encoder korunur,
    # proje protokolü için üç sınıflı head yeniden oluşturulur.
    if getattr(model.config, 'num_labels', layer.out_features) != 3 or set(raw.values()) != set(LABEL2ID):
        model.classifier = nn.Linear(layer.in_features, 3).to(layer.weight.device, dtype=layer.weight.dtype)
    else:
        old_w = layer.weight.detach().clone()
        old_b = layer.bias.detach().clone() if layer.bias is not None else None
        source = {label: idx for idx, label in raw.items()}
        with torch.no_grad():
            for target_id, label in ID2LABEL.items():
                model.classifier.weight[target_id].copy_(old_w[source[label]])
                if old_b is not None:
                    model.classifier.bias[target_id].copy_(old_b[source[label]])
    model.config.num_labels = 3
    model.num_labels = 3
    model.config.id2label = ID2LABEL.copy()
    model.config.label2id = LABEL2ID.copy()
    return model

def make_ds(df, tokenizer, text_col='input_text', label_col='label_id'):
    temp = df[[text_col, label_col]].copy().rename(columns={label_col:'labels'})
    temp[text_col] = temp[text_col].astype(str); temp['labels'] = temp['labels'].astype(int)
    ds = Dataset.from_pandas(temp.reset_index(drop=True))
    ds = ds.map(lambda b: tokenizer(b[text_col], truncation=True, padding='max_length', max_length=MAX_LENGTH), batched=True)
    keep = ['input_ids','attention_mask','labels']
    if 'token_type_ids' in ds.column_names: keep.append('token_type_ids')
    ds.set_format(type='torch', columns=keep); return ds

def trainer_tokenizer_kwargs(tokenizer):
    sig = inspect.signature(Trainer.__init__)
    return {'processing_class': tokenizer} if 'processing_class' in sig.parameters else ({'tokenizer': tokenizer} if 'tokenizer' in sig.parameters else {})

def metrics_fn(pair):
    logits, y = pair; pred = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(y,pred), 'f1_macro': f1_score(y,pred,average='macro',zero_division=0), 'f1_weighted': f1_score(y,pred,average='weighted',zero_division=0)}

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs): super().__init__(*args, **kwargs); self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs['labels']; model_inputs = {k: v for k, v in inputs.items() if k != 'labels'}; out = model(**model_inputs)
        w = self.class_weights.to(out.logits.device) if self.class_weights is not None else None
        loss = nn.CrossEntropyLoss(weight=w)(out.logits.view(-1,3), labels.view(-1))
        return (loss,out) if return_outputs else loss

def args_for(outdir):
    sig = inspect.signature(TrainingArguments.__init__)
    kw = dict(output_dir=str(outdir), learning_rate=LEARNING_RATE, per_device_train_batch_size=TRAIN_BATCH_SIZE, per_device_eval_batch_size=EVAL_BATCH_SIZE, num_train_epochs=EPOCHS, weight_decay=WEIGHT_DECAY, logging_steps=50, report_to='none', seed=RANDOM_STATE)
    if 'eval_strategy' in sig.parameters: kw['eval_strategy']='epoch'
    elif 'evaluation_strategy' in sig.parameters: kw['evaluation_strategy']='epoch'
    for k,v in [('save_strategy','epoch'),('save_total_limit',SAVE_TOTAL_LIMIT),('load_best_model_at_end',True),('metric_for_best_model','f1_macro'),('greater_is_better',True),('logging_strategy','steps')]:
        if k in sig.parameters: kw[k]=v
    if 'fp16' in sig.parameters: kw['fp16']=torch.cuda.is_available()
    return TrainingArguments(**kw)

def class_weights(df):
    c=np.bincount(df.label_id.astype(int), minlength=3); return torch.tensor([len(df)/(3*x) if x else 1.0 for x in c], dtype=torch.float) if USE_CLASS_WEIGHTS else None

In [9]:
# 1) Hazır FinancialBERT baseline: plain splitte final test ve hata analizi
split_dir = paths.PLAIN_SENTIMENT_SPLIT_V1_DIR
train_df = pd.read_parquet(split_dir/'train_df.parquet'); val_df = pd.read_parquet(split_dir/'val_df.parquet'); test_df = pd.read_parquet(split_dir/'test_df.parquet')
TEXT='input_text'; LABEL='label'; LID='label_id'
for name, df in [('train',train_df),('val',val_df),('test',test_df)]:
    df[LABEL]=df[LABEL].astype(str).str.lower().str.strip(); df[LID]=df[LID].astype(int)
    if set(df[LABEL])-set(LABEL2ID): raise ValueError(f'{name}: geçersiz label')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
base_model = align_classifier(AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)).to(DEVICE).eval()

@torch.no_grad()
def predict_model(model, tok, texts, batch_size=64):
    rows=[]
    for i in tqdm(range(0,len(texts),batch_size), desc='FinancialBERT predicting'):
        enc=tok(texts[i:i+batch_size], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
        enc={k:v.to(DEVICE) for k,v in enc.items()}; rows.append(torch.softmax(model(**enc).logits,dim=-1).cpu().numpy())
    return np.concatenate(rows)

def evaluate_frame(df, probs, name, outdir):
    y=df[LID].to_numpy(); pred=probs.argmax(1); out=df.reset_index(drop=True).copy()
    out['gold_label']=[ID2LABEL[int(x)] for x in y]; out['predicted_label']=[ID2LABEL[int(x)] for x in pred]; out['correct']=y==pred; out['confidence']=probs.max(1)
    for i,l in ID2LABEL.items(): out[f'prob_{l}']=probs[:,i]
    p,r,f1,s=precision_recall_fscore_support(y,pred,labels=[0,1,2],zero_division=0); m={'stage':name,'n_eval':len(y),'accuracy':accuracy_score(y,pred),'precision_macro':p.mean(),'recall_macro':r.mean(),'f1_macro':f1.mean(),'f1_weighted':f1_score(y,pred,average='weighted',zero_division=0),'f1_negative':f1[0],'f1_neutral':f1[1],'f1_positive':f1[2]}
    outdir.mkdir(parents=True,exist_ok=True); (outdir/f'{name}_metrics.json').write_text(json.dumps(m,indent=2),encoding='utf-8'); out.to_csv(outdir/f'{name}_predictions.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(classification_report(y,pred,labels=[0,1,2],target_names=LABELS,output_dict=True,zero_division=0)).T.to_csv(outdir/f'{name}_classification_report.csv')
    pd.DataFrame(confusion_matrix(y,pred,labels=[0,1,2]),index=[f'true_{x}' for x in LABELS],columns=[f'pred_{x}' for x in LABELS]).to_csv(outdir/f'{name}_confusion_matrix.csv')
    return m,out

def evaluate_or_load(df, model, tok, texts, name, outdir):
    pred_file=outdir/f'{name}_predictions.csv'; metrics_file=outdir/f'{name}_metrics.json'
    if pred_file.exists() and metrics_file.exists() and not FORCE_RETRAIN:
        cached=pd.read_csv(pred_file)
        same_ids=('sample_id' not in cached.columns or 'sample_id' not in df.columns or cached.sample_id.astype(str).equals(df.sample_id.astype(str)))
        if len(cached)==len(df) and same_ids:
            print('Kayıtlı test sonucu yükleniyor:', name)
            return json.loads(metrics_file.read_text(encoding='utf-8')), cached
    print('Test çalıştırılıyor:', name)
    return evaluate_frame(df, predict_model(model,tok,texts), name, outdir)

baseline_metrics, baseline_predictions = evaluate_or_load(test_df,base_model,tokenizer,test_df[TEXT].astype(str).tolist(),'ready_baseline',RESULT_ROOT/'ready_baseline')
wrong=baseline_predictions[~baseline_predictions.correct].copy(); wrong['error_transition']=wrong.gold_label+' -> '+wrong.predicted_label
print('Hazır baseline:', baseline_metrics); print('Hata tipleri:'); display(wrong['error_transition'].value_counts().to_frame('count')); display(wrong.sort_values('confidence',ascending=False).head(20))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ahmedrachid/FinancialBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

Kayıtlı test sonucu yükleniyor: ready_baseline
Hazır baseline: {'stage': 'ready_baseline', 'n_eval': 2386, 'accuracy': 0.3319362950544845, 'precision_macro': 0.4126809496236248, 'recall_macro': 0.37118054880785883, 'f1_macro': 0.28537186948298787, 'f1_weighted': 0.3326135319076861, 'f1_negative': 0.1506276150627615, 'f1_neutral': 0.37547892720306514, 'f1_positive': 0.3300090661831369}
Hata tipleri:


,count
error_transition,
neutral -> positive,1111
negative -> positive,252
positive -> neutral,77
negative -> neutral,70
neutral -> negative,46
positive -> negative,38


,sample_id,source_dataset,dataset_full_name,dataset_group,dataset_role,dataset_description,source_original_split,text,label,label_id,...,text_norm,input_text,gold_label,predicted_label,correct,confidence,prob_negative,prob_neutral,prob_positive,error_transition
695,PLAIN_SENT_0000647,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,"Patrick Harker, President of the Philly Fed, i...",neutral,1,...,"patrick harker, president of the philly fed, i...","Patrick Harker, President of the Philly Fed, i...",neutral,positive,False,0.483994,0.287278,0.228728,0.483994,neutral -> positive
1935,PLAIN_SENT_0002695,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,Edited Transcript of CB earnings conference ca...,neutral,1,...,edited transcript of cb earnings conference ca...,Edited Transcript of CB earnings conference ca...,neutral,positive,False,0.476127,0.257160,0.266713,0.476127,neutral -> positive
77,PLAIN_SENT_0002748,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,Edited Transcript of PPC earnings conference c...,neutral,1,...,edited transcript of ppc earnings conference c...,Edited Transcript of PPC earnings conference c...,neutral,positive,False,0.473753,0.274603,0.251644,0.473753,neutral -> positive
2362,PLAIN_SENT_0002726,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,Edited Transcript of LXP earnings conference c...,neutral,1,...,edited transcript of lxp earnings conference c...,Edited Transcript of LXP earnings conference c...,neutral,positive,False,0.473319,0.270351,0.256330,0.473319,neutral -> positive
2209,PLAIN_SENT_0002722,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,Edited Transcript of KWR earnings conference c...,neutral,1,...,edited transcript of kwr earnings conference c...,Edited Transcript of KWR earnings conference c...,neutral,positive,False,0.471769,0.275839,0.252392,0.471769,neutral -> positive
1804,PLAIN_SENT_0002745,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,Edited Transcript of OTEL earnings conference ...,neutral,1,...,edited transcript of otel earnings conference ...,Edited Transcript of OTEL earnings conference ...,neutral,positive,False,0.470838,0.272395,0.256767,0.470838,neutral -> positive
771,PLAIN_SENT_0002694,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,Edited Transcript of BREW earnings conference ...,neutral,1,...,edited transcript of brew earnings conference ...,Edited Transcript of BREW earnings conference ...,neutral,positive,False,0.469839,0.277850,0.252311,0.469839,neutral -> positive
874,PLAIN_SENT_0002720,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,Edited Transcript of IWSY earnings conference ...,neutral,1,...,edited transcript of iwsy earnings conference ...,Edited Transcript of IWSY earnings conference ...,neutral,positive,False,0.468699,0.280863,0.250439,0.468699,neutral -> positive
478,PLAIN_SENT_0002753,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,Edited Transcript of RRR earni

In [10]:
# Eğitim notebook'unda üretilen sonuçları yükle
baseline_metrics = json.loads((RESULT_ROOT/'ready_baseline'/'ready_baseline_metrics.json').read_text(encoding='utf-8')) if (RESULT_ROOT/'ready_baseline'/'ready_baseline_metrics.json').exists() else {}
target_tok = AutoTokenizer.from_pretrained(TARGET_ROOT/'final_model')
target_model = AutoModelForSequenceClassification.from_pretrained(TARGET_ROOT/'final_model').to(DEVICE).eval()
target_metrics, target_predictions = evaluate_or_load(test_df, target_model, target_tok, test_df[TEXT].astype(str).tolist(), 'target_internal_test', RESULT_ROOT/'target_dataset')
cons_test = pd.read_csv(DATA_ROOT/'splits'/'consensus_twitter_phrasebank_allagree_v1'/'internal_test.csv')
cons_test['input_text'] = cons_test['input_text'].astype(str)
cons_tok = AutoTokenizer.from_pretrained(CONSENSUS_ROOT/'final_model')
cons_model = AutoModelForSequenceClassification.from_pretrained(CONSENSUS_ROOT/'final_model').to(DEVICE).eval()
cons_metrics, cons_predictions = evaluate_or_load(cons_test, cons_model, cons_tok, cons_test.input_text.tolist(), 'consensus_internal_test', RESULT_ROOT/'consensus')


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Kayıtlı test sonucu yükleniyor: target_internal_test


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Kayıtlı test sonucu yükleniyor: consensus_internal_test


In [11]:
from thesis_utils.data_io import clean_string_series, make_unique_columns, normalize_column_name, extract_batch_number, find_header_row as io_find_header_row, read_raw_table_no_header, natural_sort_key

def normalize_full_external(df, gold_candidates, dataset_name):
    df = df.copy()
    text_col = next((c for c in ['text_en','text','source_text','headline','title','annotation_text'] if c in df.columns), None)
    if text_col is None: raise ValueError(f'{dataset_name}: text column missing')
    df['input_text'] = clean_string_series(df[text_col])
    df['gold_label'] = pd.NA
    for col in gold_candidates:
        if col in df.columns:
            df['gold_label'] = df['gold_label'].fillna(clean_string_series(df[col]).str.lower())
    df = df[df['input_text'].notna() & df['gold_label'].isin(LABELS)].copy().reset_index(drop=True)
    if 'sample_id' not in df.columns:
        df.insert(0, 'sample_id', [f'{dataset_name}_{i:06d}' for i in range(len(df))])
    else:
        df['sample_id'] = clean_string_series(df['sample_id'])
        missing = df['sample_id'].isna()
        df.loc[missing, 'sample_id'] = [f'{dataset_name}_{i:06d}' for i in df.index[missing]]
    df['label_id'] = df['gold_label'].map(LABEL2ID).astype(int)
    df['dataset'] = dataset_name
    return df

def load_full_external():
    sp_dir = paths.SP500_HUMAN_REVIEW_BATCHES_DIR
    sp_files = sorted([p for p in sp_dir.rglob('SP500_annotation_batch_*.xlsx') if not p.name.startswith('~$')], key=lambda p:(extract_batch_number(p),p.name.lower()))
    sp_frames = []
    for p in sp_files:
        raw = pd.read_excel(p, header=None, dtype=object)
        h = io_find_header_row(raw, required_cols=('annotation_id','sample_id','text_en'), min_hits=2, max_scan_rows=10)
        h = 0 if h is None else h
        d = pd.read_excel(p, header=h, dtype=object)
        d.columns = [normalize_column_name(c) for c in d.columns]
        sp_frames.append(d.dropna(how='all'))
    sp = normalize_full_external(pd.concat(sp_frames, ignore_index=True), ['final_label','chatgpt_label'], 'sp500_external')
    if len(sp_files) != 106 or len(sp) != 1060:
        raise ValueError(f'S&P full set mismatch: files={len(sp_files)}, rows={len(sp)}')

    syn_files = sorted([p for pat in ['positive_batch_*.xlsx','negative_batch_*.xlsx','neutral_batch_*.xlsx'] for p in paths.SYNTHETIC_NEWS_DIR.glob(pat) if not p.name.startswith('~$')], key=natural_sort_key)
    syn_frames = []
    for p in syn_files:
        xls = pd.ExcelFile(p)
        syn_frames.append(pd.read_excel(p, sheet_name='Data' if 'Data' in xls.sheet_names else xls.sheet_names[0]))
    syn = normalize_full_external(pd.concat(syn_frames, ignore_index=True), ['label'], 'synthetic_external')
    if len(syn) != 3000: raise ValueError(f'Synthetic full set mismatch: rows={len(syn)}')

    def read_reuters_file(path):
        raw = read_raw_table_no_header(path).dropna(how='all').copy()
        h = io_find_header_row(raw)
        if h is None:
            d = pd.read_excel(path, dtype=object)
        else:
            d = raw.iloc[h+1:].copy(); d.columns = make_unique_columns(raw.iloc[h].tolist())
        d.columns = [normalize_column_name(c) for c in d.columns]
        return d.dropna(how='all').dropna(axis=1, how='all')
    # CSV'ler tam text_en içerir; XLSX'ler tam review label içerir. Annotation ID ile birmalıyoruz.
    csv_files = sorted([p for p in paths.REUTERS_ANNOTATION_DIR.rglob('*.csv') if 'annotation_batch_' in p.name.lower() and not p.name.startswith('~$')], key=lambda p:(extract_batch_number(p),p.name.lower()))
    xlsx_files = sorted([p for p in paths.REUTERS_ANNOTATION_DIR.rglob('*.xlsx') if 'annotation_batch_' in p.name.lower() and not p.name.startswith('~$')], key=lambda p:(extract_batch_number(p),p.name.lower()))
    csv_df = pd.concat([pd.read_csv(p, encoding='utf-8-sig', dtype=object) for p in csv_files], ignore_index=True)
    csv_df.columns = [normalize_column_name(c) for c in csv_df.columns]
    xlsx_df = pd.concat([read_reuters_file(p) for p in xlsx_files], ignore_index=True)
    xlsx_df.columns = [normalize_column_name(c) for c in xlsx_df.columns]
    xlsx_df = xlsx_df[xlsx_df['annotation_id'].notna()].reset_index(drop=True)
    if len(csv_df) != len(xlsx_df): raise ValueError(f'Reuters CSV/XLSX row mismatch: {len(csv_df)} vs {len(xlsx_df)}')
    reu_merged = csv_df.copy()
    reu_merged['chatgpt_label'] = xlsx_df['chatgpt_label'].reset_index(drop=True)
    reu = normalize_full_external(reu_merged, ['final_label','chatgpt_label'], 'reuters_external')
    if len(reu) != 5000: raise ValueError(f'Reuters full set mismatch: rows={len(reu)}')
    return {'sp500_external':sp, 'synthetic_external':syn, 'reuters_external':reu}

external = load_full_external()
external_rows = []

def eval_external(model_key, model_dir):
    tok = AutoTokenizer.from_pretrained(model_dir)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_dir)
    if model_key == 'ready': mdl = align_classifier(mdl)
    mdl = mdl.to(DEVICE).eval()
    rows = []
    for dataset, df in external.items():
        outdir = RESULT_ROOT/'external'/model_key; outdir.mkdir(parents=True, exist_ok=True)
        name = f'{model_key}_{dataset}'
        pred_file = outdir/f'{name}_predictions.csv'
        if pred_file.exists() and not FORCE_RETRAIN:
            pred = pd.read_csv(pred_file)
            if pred.sample_id.astype(str).tolist() != df.sample_id.astype(str).tolist():
                raise RuntimeError(f'{name}: mevcut prediction sample_id sırası uyuşmuyor')
            m = json.loads((outdir/f'{name}_metrics.json').read_text(encoding='utf-8'))
        else:
            probs = predict_model(mdl, tok, df.input_text.tolist())
            tmp = df.rename(columns={'gold_label':'label'}).copy()
            m, _ = evaluate_frame(tmp, probs, name, outdir)
        m.update({'model':model_key, 'dataset':dataset, 'seed':RANDOM_STATE})
        rows.append(m)
    del mdl, tok; gc.collect()
    return rows

for model_key, model_dir in {'ready':MODEL_NAME, 'target':TARGET_ROOT/'final_model', 'consensus':CONSENSUS_ROOT/'final_model'}.items():
    external_rows.extend(eval_external(model_key, model_dir))
summary=pd.DataFrame([baseline_metrics,target_metrics,cons_metrics]+external_rows)
summary.to_csv(RESULT_ROOT/'financialbert_full_summary.csv',index=False,encoding='utf-8-sig')
display(summary.round(4))
print('Tüm çıktı kökü:', RESULT_ROOT)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ahmedrachid/FinancialBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,stage,n_eval,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,f1_negative,f1_neutral,f1_positive,model,dataset,seed
0,ready_baseline,2386,0.3319,0.4127,0.3712,0.2854,0.3326,0.1506,0.3755,0.3300,NaN,NaN,NaN
1,target_internal_test,2386,0.8223,0.7612,0.7731,0.7669,0.8234,0.6713,0.8821,0.7472,NaN,NaN,NaN
2,consensus_internal_test,1630,0.8368,0.8084,0.8148,0.8115,0.8374,0.7543,0.8881,0.7920,NaN,NaN,NaN
3,ready_sp500_external,1060,0.3255,0.2515,0.3369,0.1857,0.1822,0.0000,0.4835,0.0736,ready,sp500_external,42.0
4,ready_synthetic_external,3000,0.3537,0.2459,0.3537,0.2635,0.2635,0.0000,0.4805,0.3100,ready,synthetic_external,42.0
5,ready_reuters_external,5000,0.2504,0.3311,0.3266,0.1799,0.1647,0.0013,0.3574,0.1810,ready,reuters_external,42.0
6,target_sp500_external,1060,0.6264,0.6393,0.6234,0.6210,0.6232,0.5730,0.6429,0.6472,target,sp500_external,42.0
7,target_synthetic_external,3000,0.9857,0.9858,0.9857,0.9857,0.9857,0.9813,0.9940,0.9817,target,synthetic_external,42.0
8,target_reuters_external,5000,0.6118,0.7050,0.6571,0.6220,0.6283,0.7287,0.5283,0.6091,target,reuters_external,42.0
9,consensus_sp500_external,1060,0.6443,0.6478,0.6397,0.6391,0.6418,0.5874,0.6566,0.6732,consensus,sp500_external,42.0


Tüm çıktı kökü: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\outputs\finetuned_model_results\financialbert_full_pipeline_financialbert_seed42_v2


In [12]:
# 5) Eşleştirilmiş istatistiksel karşılaştırma: hazır vs hedef vs consensus
from math import comb
def exact_mcnemar(a,b):
    x=np.asarray(a); y=np.asarray(b); d=int(np.sum((x==1)&(y==0))+np.sum((x==0)&(y==1)))
    if d==0: return 0,0,1.0
    bo=int(np.sum((x==1)&(y==0))); co=int(np.sum((x==0)&(y==1))); k=min(bo,co)
    return bo,co,min(1.0,2*sum(comb(d,j) for j in range(k+1))/(2**d))
stat_rows=[]
for dataset in external:
    frames={}
    for key in ['ready','target','consensus']:
        p=RESULT_ROOT/'external'/key/f'{key}_{dataset}_predictions.csv'
        if p.exists(): frames[key]=pd.read_csv(p)
    if len(frames)<3: raise RuntimeError(f'{dataset}: ready/target/consensus prediction eksik')
    ref=next(iter(frames)); base=frames[ref]
    for other,df in frames.items():
        if other==ref: continue
        if not base.sample_id.astype(str).equals(df.sample_id.astype(str)): raise ValueError(f'{dataset}: sample_id eşleşmiyor')
        y=(base.gold_label==base.predicted_label).astype(int).to_numpy(); z=(df.gold_label==df.predicted_label).astype(int).to_numpy(); bo,co,pv=exact_mcnemar(y,z)
        stat_rows.append({'dataset':dataset,'reference':ref,'comparison':other,'n':len(df),'reference_accuracy':float(y.mean()),'comparison_accuracy':float(z.mean()),'reference_only_correct':bo,'comparison_only_correct':co,'mcnemar_p_value':pv})
stat_df=pd.DataFrame(stat_rows)
if stat_df.empty: raise RuntimeError('McNemar sonucu boş; prediction dosyaları eşleşmedi')
stat_df.to_csv(RESULT_ROOT/'financialbert_mcnemar_comparisons.csv',index=False,encoding='utf-8-sig'); display(stat_df.round(6))

# 6) Mevcut fine-tuned modellerle karşılaştırma (tek seed; ortalama/std yok)
existing_rows=[]
existing_models={'bert_base_uncased':paths.MODEL_CHECKPOINT_ROOT/'bert_base_uncased'/'final_model','distilbert_base_uncased':paths.MODEL_CHECKPOINT_ROOT/'distilbert_base_uncased'/'final_model','roberta_base':paths.MODEL_CHECKPOINT_ROOT/'roberta_base'/'final_model'}
for model_key, model_dir in existing_models.items():
    if not (model_dir/'config.json').exists(): print(f'{model_key}: checkpoint yok, atlanıyor'); continue
    tok=AutoTokenizer.from_pretrained(model_dir); mdl=AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE).eval()
    m,_=evaluate_or_load(test_df,mdl,tok,test_df[TEXT].astype(str).tolist(),f'{model_key}_target_test',RESULT_ROOT/'existing'/model_key); m.update({'model':model_key,'dataset':'target_internal_test','seed':RANDOM_STATE}); existing_rows.append(m)
    for dataset,df in external.items():
        tmp=df.rename(columns={'gold_label':'label'}).copy(); m,_=evaluate_or_load(tmp,mdl,tok,df.input_text.tolist(),f'{model_key}_{dataset}',RESULT_ROOT/'existing'/model_key); m.update({'model':model_key,'dataset':dataset,'seed':RANDOM_STATE}); existing_rows.append(m)
    del mdl,tok; gc.collect()
existing_df=pd.DataFrame(existing_rows); all_models_df=pd.concat([summary,existing_df],ignore_index=True); all_models_df.to_csv(RESULT_ROOT/'all_models_comparison_seed42.csv',index=False,encoding='utf-8-sig'); display(all_models_df.round(4))

,dataset,reference,comparison,n,reference_accuracy,comparison_accuracy,reference_only_correct,comparison_only_correct,mcnemar_p_value
0,sp500_external,ready,target,1060,0.325472,0.626415,85,404,0.0
1,sp500_external,ready,consensus,1060,0.325472,0.644340,99,437,0.0
2,synthetic_external,ready,target,3000,0.353667,0.985667,4,1900,0.0
3,synthetic_external,ready,consensus,3000,0.353667,0.984667,1,1894,0.0
4,reuters_external,ready,target,5000,0.250400,0.611800,308,2115,0.0
5,reuters_external,ready,consensus,5000,0.250400,0.626000,341,2219,0.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Kayıtlı test sonucu yükleniyor: bert_base_uncased_target_test
Kayıtlı test sonucu yükleniyor: bert_base_uncased_sp500_external
Kayıtlı test sonucu yükleniyor: bert_base_uncased_synthetic_external
Kayıtlı test sonucu yükleniyor: bert_base_uncased_reuters_external


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Kayıtlı test sonucu yükleniyor: distilbert_base_uncased_target_test
Kayıtlı test sonucu yükleniyor: distilbert_base_uncased_sp500_external
Kayıtlı test sonucu yükleniyor: distilbert_base_uncased_synthetic_external
Kayıtlı test sonucu yükleniyor: distilbert_base_uncased_reuters_external


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

... (5 outputs left out)

FinancialBERT predicting:   0%|          | 0/47 [00:00<?, ?it/s]

Test çalıştırılıyor: roberta_base_reuters_external


FinancialBERT predicting:   0%|          | 0/79 [00:00<?, ?it/s]

,stage,n_eval,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,f1_negative,f1_neutral,f1_positive,model,dataset,seed
0,ready_baseline,2386,0.3319,0.4127,0.3712,0.2854,0.3326,0.1506,0.3755,0.3300,NaN,NaN,NaN
1,target_internal_test,2386,0.8223,0.7612,0.7731,0.7669,0.8234,0.6713,0.8821,0.7472,NaN,NaN,NaN
2,consensus_internal_test,1630,0.8368,0.8084,0.8148,0.8115,0.8374,0.7543,0.8881,0.7920,NaN,NaN,NaN
3,ready_sp500_external,1060,0.3255,0.2515,0.3369,0.1857,0.1822,0.0000,0.4835,0.0736,ready,sp500_external,42.0
4,ready_synthetic_external,3000,0.3537,0.2459,0.3537,0.2635,0.2635,0.0000,0.4805,0.3100,ready,synthetic_external,42.0
5,ready_reuters_external,5000,0.2504,0.3311,0.3266,0.1799,0.1647,0.0013,0.3574,0.1810,ready,reuters_external,42.0
6,target_sp500_external,1060,0.6264,0.6393,0.6234,0.6210,0.6232,0.5730,0.6429,0.6472,target,sp500_external,42.0
7,target_synthetic_external,3000,0.9857,0.9858,0.9857,0.9857,0.9857,0.9813,0.9940,0.9817,target,synthetic_external,42.0
8,target_reuters_external,5000,0.6118,0.7050,0.6571,0.6220,0.6283,0.7287,0.5283,0.6091,target,reuters_external,42.0
9,consensus_sp500_external,1060,0.6443,0.6478,0.6397,0.6391,0.6418,0.5874,0.6566,0.6732,consensus,sp500_external,42.0


## Çıktılar ve tez kullanımı

- Checkpointler: `checkpoints/financial_sentiment_multi_model/financialbert_full_pipeline/{target_dataset,consensus}/checkpoints/`
- Final modeller: aynı klasörlerde `final_model/`
- Metrikler, prediction CSV'leri, classification report ve confusion matrix: `outputs/finetuned_model_results/financialbert_full_pipeline/`
- Ana karşılaştırma: `financialbert_full_summary.csv`

Eğitim yarıda kalırsa aynı hücre yeniden çalıştırıldığında son epoch checkpoint'i `get_last_checkpoint` ile bulunur ve `resume_from_checkpoint` ile devam eder. Baştan eğitim için `FORCE_RETRAIN = True` yapılmalıdır.